# অধ্যায় ৬: পাইপলাইন
## পাঠ ৬.১: পাইপলাইন তৈরি

আজ আমরা শিখব কীভাবে মেশিন লার্নিং পাইপলাইন তৈরি করতে হয়। পাইপলাইন আমাদের মডেল ট্রেনিং প্রক্রিয়াকে সুশৃঙ্খল এবং স্বয়ংক্রিয় করে তোলে।

### A. গল্প: একটি কারখানা

একটি পিৎজা তৈরির কারখানা কল্পনা করো। সেখানে একটি কনভেয়র বেল্ট আছে:
১. ময়দা তৈরি → ২. সস দেওয়া → ৩. টপিংস দেওয়া → ৪. বেক করা → ৫. প্যাকেট করা

প্রতিটি স্টেপ একটি 'বক্স'। সবগুলো স্টেপ মিলে একটি 'পাইপলাইন'। প্রথমে ময়দা যায়, শেষে প্যাকেট করা পিৎজা বের হয়।

মেশিন লার্নিং পাইপলাইনও ঠিক সেরকম—প্রথমে ডেটা যায়, তারপর বিভিন্ন ট্রান্সফর্মেশন হয় (স্কেলিং, এনকোডিং), শেষে একটি ক্লাসিফায়ার বা রিগ্রেসর প্রেডিকশন করে।

### B. পাইপলাইন কেন দরকারি?

১. **সুবিধা:** একটি `fit` এবং `predict` কলেই সবকিছু হয়ে যায়
২. **ডেটা লিকেজ প্রতিরোধ:** পাইপলাইন নিশ্চিত করে যে ট্রান্সফর্মেশন শুধু ট্রেনিং ডেটায় ফিট হয়
৩. **গ্রিড সার্চ সহজ:** পাইপলাইনের প্যারামিটারগুলো সহজেই গ্রিড সার্চ করা যায়
৪. **প্রোডাকশনে সহজ:** একক অবজেক্ট হিসেবে সেভ এবং লোড করা যায়

### C. Pipeline vs make_pipeline

দুটি উপায়ে পাইপলাইন তৈরি করা যায়:

**Pipeline:** নাম দিয়ে স্টেপগুলো নির্ধারণ করি।
```python
Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier())])
```

**make_pipeline:** নাম স্বয়ংক্রিয়ভাবে দেওয়া হয়।
```python
make_pipeline(StandardScaler(), KNeighborsClassifier())
```

দুটির কাজ একই। Pipeline বেশি নিয়ন্ত্রণ দেয়, make_pipeline সহজ।

In [1]:
# প্রয়োজনীয় লাইব্রেরি
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import accuracy_score, classification_report

# ডেটা লোড করি
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print('Breast Cancer ডেটাসেট:')
print(f'  নমুনা: {X.shape[0]}, ফিচার: {X.shape[1]}')
print(f'  ক্লাস: ক্যান্সার নয়={np.sum(y==0)}, ক্যান্সার={np.sum(y==1)}')

Breast Cancer ডেটাসেট:
  নমুনা: 569, ফিচার: 30
  ক্লাস: ক্যান্সার নয়=212, ক্যান্সার=357


### D. প্রথম পাইপলাইন তৈরি

এখন আমরা একটি পাইপলাইন তৈরি করব যেখানে প্রথমে StandardScaler দিয়ে ফিচার স্কেল করবো, তারপর KNN ক্লাসিফায়ার ব্যবহার করবো।

In [2]:
# Pipeline ব্যবহার করি
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

# পাইপলাইন ফিট করি (এক লাইনে সব!)  
pipe.fit(X_train, y_train)

# প্রেডিক্ট করি
y_pred = pipe.predict(X_test)
print('পাইপলাইন ব্যবহার করে KNN:')
print(f'  Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'  স্কেলিং + KNN একসাথে কাজ করছে!')
print('\nপাইপলাইনের স্টেপগুলো:')
for name, step in pipe.named_steps.items():
    print(f'  {name}: {step.__class__.__name__}')

পাইপলাইন ব্যবহার করে KNN:
  Accuracy: 0.9591
  স্কেলিং + KNN একসাথে কাজ করছে!

পাইপলাইনের স্টেপগুলো:
  scaler: StandardScaler
  knn: KNeighborsClassifier


### E. make_pipeline ব্যবহার

make_pipeline দিয়ে পাইপলাইন তৈরি আরও সহজ। নামগুলো স্বয়ংক্রিয়ভাবে দেওয়া হয়।

In [3]:
# make_pipeline ব্যবহার করি
pipe2 = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
pipe2.fit(X_train, y_train)
print('make_pipeline স্টেপের নাম:')
print('  ', pipe2.named_steps)
print(f'\nAccuracy: {pipe2.score(X_test, y_test):.4f}')
print('\nmake_pipeline কাজ করে Pipeline-এর মতোই, তবে নাম স্বয়ংক্রিয়!')

make_pipeline স্টেপের নাম:
   {'standardscaler': StandardScaler(), 'kneighborsclassifier': KNeighborsClassifier()}

Accuracy: 0.9591

make_pipeline কাজ করে Pipeline-এর মতোই, তবে নাম স্বয়ংক্রিয়!


### F. লম্বা পাইপলাইন: স্কেলিং + PCA + ক্লাসিফায়ার

এখন আমরা তিন-স্টেপের পাইপলাইন তৈরি করব:
১. StandardScaler (ফিচার স্কেলিং)
২. PCA (ডাইমেনশনালিটি রিডাকশন)
৩. KNeighborsClassifier (ক্লাসিফিকেশন)

PCA (Principal Component Analysis) জটিল ফিচারগুলোকে কম সংখ্যক গুরুত্বপূর্ণ ফিচারে রূপান্তর করে। এটা ডাইমেনশনালিটি রিডাকশনের একটি পদ্ধতি।

In [4]:
# তিন-স্টেপ পাইপলাইন
pipe3 = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=5)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

pipe3.fit(X_train, y_train)
print('তিন-স্টেপ পাইপলাইন:')
print(f'  Accuracy: {pipe3.score(X_test, y_test):.4f}')
print(f'  PCA 30টি ফিচার থেকে {pipe3.named_steps["pca"].n_components_}টি কম্পোনেন্টে নিয়েছে')

# cross_val_score দিয়ে মূল্যায়ন
cv_scores = cross_val_score(pipe3, X_train, y_train, cv=5)
print(f'  Cross-val Accuracy: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})')

তিন-স্টেপ পাইপলাইন:
  Accuracy: 0.9591
  PCA 30টি ফিচার থেকে 5টি কম্পোনেন্টে নিয়েছে
  Cross-val Accuracy: 0.9572 (+/- 0.0151)


### G. SVM-এর জন্য পাইপলাইন

SVM (Support Vector Machine) স্কেলিং-এর প্রতি খুব সংবেদনশীল। পাইপলাইনে স্কেলিং যোগ করা খুবই গুরুত্বপূর্ণ।

In [5]:
# SVM পাইপলাইন
pipe_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
)

pipe_svm.fit(X_train, y_train)
print(f'SVM পাইপলাইন Accuracy: {pipe_svm.score(X_test, y_test):.4f}')

# SVM পাইপলাইন (স্কেলিং ছাড়া) - তুলনা
svm_no_scale = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_no_scale.fit(X_train, y_train)
print(f'SVM (স্কেলিং ছাড়া) Accuracy: {svm_no_scale.score(X_test, y_test):.4f}')
print('\n→ স্কেলিং SVM-এর জন্য খুবই গুরুত্বপূর্ণ!')
print('→ পাইপলাইন নিশ্চিত করে স্কেলিং সবসমই সঠিকভাবে প্রয়োগ হয়')

SVM পাইপলাইন Accuracy: 0.9766
SVM (স্কেলিং ছাড়া) Accuracy: 0.9415

→ স্কেলিং SVM-এর জন্য খুবই গুরুত্বপূর্ণ!
→ পাইপলাইন নিশ্চিত করে স্কেলিং সবসমই সঠিকভাবে প্রয়োগ হয়


### H. পাইপলাইনের সুবিধা: কোড কম, নির্ভুলতা বেশি

পাইপলাইন ছাড়া যা করতে হতো:
```
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # মনে রাখতে হবে transform ব্যবহার করতে!
model = KNeighborsClassifier()
model.fit(X_train_scaled, y_train)
model.predict(X_test_scaled)
```

পাইপলাইন দিয়ে:
```
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
pipe.fit(X_train, y_train)
pipe.predict(X_test)
```

পাইপলাইন স্বয়ংক্রিয়ভাবে ট্রেন ডেটায় `fit_transform` এবং টেস্ট ডেটায় `transform` ব্যবহার করে—আমাদের কিছু মনে রাখতে হয় না!

### I. তুমি কি বুঝতে পেরেছ?

**প্রশ্ন ১:** পাইপলাইন ব্যবহার করার তিনটি সুবিধা বলো।

**প্রশ্ন ২:** Pipeline এবং make_pipeline-এর মধ্যে পার্থক্য কী?

**প্রশ্ন ৩:** পাইপলাইন কীভাবে ডেটা লিকেজ প্রতিরোধ করে?

**প্রশ্ন ৪:** SVM-এর জন্য স্কেলিং কেন গুরুত্বপূর্ণ?

### J. সারসংক্ষেপ

আজ আমরা শিখলাম:
✅ পাইপলাইন মেশিন লার্নিং ওয়ার্কফ্লোকে সুশৃঙ্খল করে
✅ Pipeline([('নাম', স্টেপ)]) এবং make_pipeline(স্টেপ) দিয়ে তৈরি করা যায়
✅ পাইপলাইন স্বয়ংক্রিয়ভাবে fit/transform সঠিকভাবে প্রয়োগ করে
✅ পাইপলাইন ডেটা লিকেজ প্রতিরোধ করে
✅ স্কেলিং, PCA, ক্লাসিফায়ার—সবকিছু এক পাইপলাইনে চেইন করা যায়

পরবর্তী পাঠে আমরা পাইপলাইনের সাথে গ্রিড সার্চ এবং ডেটা লিকেজ নিয়ে বিস্তারিত শিখব!